<a href="https://colab.research.google.com/github/kumarmohit0911/Algerian_forest_fire/blob/main/hyperparam_tuning_using_optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install optuna

In [4]:
# calling all the dependencies
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset,DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [5]:
#for reproducibility
torch.manual_seed(42)

In [6]:
# loading the data set
df = pd.read_csv("/content/Gas_Sensors_Measurements (2).csv")

In [7]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [8]:
df = df.drop(["Serial Number","Corresponding Image Name"],axis = 1)


In [9]:
for i in df['Gas']:
  if i == 'NoGas':
    df['Gas'] = df['Gas'].replace(i,0)
  if i == 'Perfume':
    df['Gas'] = df['Gas'].replace(i,1)
  if i == 'Smoke':
    df['Gas'] = df['Gas'].replace(i,2)
  if i == 'Mixture':
    df['Gas'] = df['Gas'].replace(i,3)

/tmp/ipython-input-512672298.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Gas'] = df['Gas'].replace(i,3)


In [10]:
X = df[['MQ2','MQ3','MQ5', 'MQ6',	'MQ7',	'MQ8',	'MQ135']]
y = df['Gas']

In [11]:
# scaling the features

sc = StandardScaler()
X = sc.fit_transform(X)

In [12]:
# Train test Split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state=42)

In [13]:
# creating custom dataset

class CustomDataset(Dataset):
  def __init__(self,features, labels):
    self.features = torch.tensor(features, dtype = torch.float32)
    self.labels = torch.tensor(labels, dtype = torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    return self.features[index],self.labels[index]


In [14]:
train_dataset = CustomDataset(X_train,y_train.values)
test_dataset = CustomDataset(X_test,y_test.values)

In [15]:
class MyNN(nn.Module):
  def __init__(self, input_dim, output_dim,
               num_hidden_layer, neurons_per_layer,Dropout_rate):
    super().__init__()
    layers = []
    for i in range(num_hidden_layer):
      layers.append(nn.Linear(input_dim,neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU()),
      layers.append(nn.Dropout(p=Dropout_rate))
      input_dim = neurons_per_layer
    layers.append(nn.Linear(neurons_per_layer,output_dim))
    self.model = nn.Sequential(*layers)
  def forward(self,x):
    return self.model(x)

In [16]:
# ready the objective function
def objective(trial):
  # hyperparameter value for search space
  num_hidden_layer = trial.suggest_int("num_hidden_layers",1,5)
  neurons_per_layer = trial.suggest_int("neurons_per_layer",8,128,step=8)
  epochs = trial.suggest_int("epochs",50,200,step=10)
  learning_rate = trial.suggest_float("learning_rate",1e-5,1e-1,log = True)
  Dropout_rate = trial.suggest_float("Dropout_rate",0.0,0.5,step=0.1)
  batch_size = trial.suggest_categorical("batch_size",[16,32,64,128])
  optimizer = trial.suggest_categorical("optimizer",["Adam","SGD","RMSprop"])
  weight_decay = trial.suggest_float("weight_decay",1e-5,1e-1,log = True)
  #model initialisation
  input_dim = 7
  output_dim = 4
  model = MyNN(input_dim,output_dim,num_hidden_layer,neurons_per_layer,Dropout_rate)
  model.to(device)
  # Creating train and test dataloader
  train_loader = DataLoader(train_dataset,batch_size = batch_size, shuffle =True)
  test_loader = DataLoader(test_dataset,batch_size = batch_size,shuffle = False)


  #optimmizer selectiom
  criterion = nn.CrossEntropyLoss()
  # optimizer = optim.SGD(model.parameters(),lr = learning_rate,weight_decay = 1e-4)
  if optimizer == "SGD":
    optimizer = optim.SGD(model.parameters(),lr = learning_rate,weight_decay = weight_decay)
  elif optimizer == "Adam":
    optimizer = optim.Adam(model.parameters(),lr = learning_rate,weight_decay = weight_decay)
  else:
    optimizer = optim.RMSprop(model.parameters(),lr = learning_rate,weight_decay = weight_decay)

  #training loop
  for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features,batch_labels in train_loader:

      #modifying training loop data to gpu
      batch_features,batch_labels = batch_features.to(device), batch_labels.to(device)


      #forward pass
      outputs = model(batch_features)

      #calculate loss
      loss = criterion(outputs,batch_labels)

      #backward pass
      optimizer.zero_grad()
      loss.backward()

      #update weights
      optimizer.step()

  #evaluation
  model.eval()
  #evaluation code over test data
  #Test Code
  total = 0
  correct = 0
  with torch.no_grad():
    for batch_features,batch_labels in test_loader:
      batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
      outputs = model(batch_features)
      _,predicted = torch.max(outputs.data,1)
      total += batch_labels.shape[0]
      correct += (predicted == batch_labels).sum().item()

  accuracy = (correct/total)
  return accuracy

In [17]:
import optuna
study = optuna.create_study(direction = 'maximize')


[I 2026-01-02 19:09:12,192] A new study created in memory with name: no-name-fd186d6e-6ad5-43a7-b34b-1b4914ebf9f6


In [18]:
study.optimize(objective, n_trials= 10)

[I 2026-01-02 19:09:41,277] Trial 0 finished with value: 0.934375 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 112, 'epochs': 130, 'learning_rate': 7.503219004450853e-05, 'Dropout_rate': 0.0, 'batch_size': 64, 'optimizer': 'Adam', 'weight_decay': 1.575877289652186e-05}. Best is trial 0 with value: 0.934375.
[I 2026-01-02 19:10:12,338] Trial 1 finished with value: 0.940625 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 88, 'epochs': 200, 'learning_rate': 0.0003288422605380726, 'Dropout_rate': 0.2, 'batch_size': 64, 'optimizer': 'Adam', 'weight_decay': 0.00018050384624383663}. Best is trial 1 with value: 0.940625.
[I 2026-01-02 19:12:11,000] Trial 2 finished with value: 0.9140625 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 128, 'epochs': 120, 'learning_rate': 0.02047368048763109, 'Dropout_rate': 0.0, 'batch_size': 16, 'optimizer': 'RMSprop', 'weight_decay': 0.0004008073829120423}. Best is trial 1 with value: 0.940625.
[I 2026-01-02 19:12:

In [19]:
study.best_value

0.9515625

In [20]:
study.best_params

{'num_hidden_layers': 3,
 'neurons_per_layer': 72,
 'epochs': 50,
 'learning_rate': 0.0036169817731897262,
 'Dropout_rate': 0.0,
 'batch_size': 64,
 'optimizer': 'Adam',
 'weight_decay': 9.491631946039704e-05}